In [ ]:
# مرحله 1: کلون کردن مخزن
!git clone https://github.com/mohammadnabia/DA_nnUNet.git

# مرحله 2: وارد فولدر پروژه
%cd DA_nnUNet

# مرحله 3: نصب پروژه در حالت editable
!pip install -e .


Cloning into 'DA_nnUNet'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 297 (delta 41), reused 267 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (297/297), 2.58 MiB | 1.94 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/DA_nnUNet
Obtaining file:///content/DA_nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/BraTS-peds2023/BraTS-PEDs-2023.zip" -d /content/BraTS_PEDs_2023


In [ ]:
import os
import shutil
import json
import glob

source_dir = "/content/BraTS_PEDs_2023/ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData"
target_dir = "/content/imagesTs"
os.makedirs(target_dir, exist_ok=True)

with open('/content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune/splits_final.json', 'r') as f:
    splits = json.load(f)

val_cases = splits[0]['val']  # Fold 0

for case in val_cases:
    case_path = os.path.join(source_dir, case)
    if not os.path.isdir(case_path):
        continue
    try:
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1n.nii.gz"))[0], os.path.join(target_dir, f"{case}_0000.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1c.nii.gz"))[0], os.path.join(target_dir, f"{case}_0001.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2w.nii.gz"))[0], os.path.join(target_dir, f"{case}_0002.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2f.nii.gz"))[0], os.path.join(target_dir, f"{case}_0003.nii.gz"))
    except IndexError:
        print(f"⚠️ Missing modalities for case: {case}")

print("✅ Validation fold 0 cases copied to imagesTs.")


✅ Validation fold 0 cases copied to imagesTs.


In [ ]:
!ls /content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune


dataset.json	     fold_0_epoch_30.pth  splits_final.json
fold_0_epoch_10.pth  fold_0_epoch_40.pth
fold_0_epoch_20.pth  nnUNetPlans.json


In [ ]:
!mkdir -p /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0

# کپی checkpoint
!cp "/content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune/fold_0_epoch_40.pth" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth

# کپی plans.json
!cp "/content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune/nnUNetPlans.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/plans.json

# کپی dataset.json
!cp "/content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune/dataset.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/

# کپی splits_final.json
!cp "/content/drive/MyDrive/40_epoch_training_on_bratspeds_finetune/splits_final.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/


In [ ]:
import os

os.environ['nnUNet_raw'] = "/content/nnUNet_raw_baseline"
os.environ['nnUNet_preprocessed'] = "/content/nnUNet_preprocessed_baseline"
os.environ['nnUNet_results'] = "/content/nnUNet_results_baseline"


In [ ]:
!mkdir -p /content/nnUNet_raw_baseline
!mkdir -p /content/nnUNet_preprocessed_baseline
!mkdir -p /content/nnUNet_results_baseline


In [ ]:
import re

file_path = '/content/DA_nnUNet/nnunetv2/inference/predict_from_raw_data.py'
with open(file_path, 'r') as f:
    code = f.read()

# تغییر خط prediction
code = re.sub(r'prediction, _ = self\.network\(x\)', 'prediction = self.network(x)', code)

with open(file_path, 'w') as f:
    f.write(code)

print("✅ خط network(x) اصلاح شد!")


✅ خط network(x) اصلاح شد!


In [ ]:
!nnUNetv2_predict \
  -i /content/imagesTs \
  -o /content/pred_fold0_noTTA \
  -d 139 \
  -c 3d_fullres \
  -f 0 \
  -tr nnUNetTrainer_TL_FTen_Custom_100epochs \
  --disable_tta



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 20 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 20 cases that I would like to predict

Predicting BraTS-PED-00008-000:
perform_everything_on_device: True
100% 8/8 [00:01<00:00,  5.82it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00008-000

Predicting BraTS-PED-00021-000:
perform_everything_on_device: True
100% 8/8 [00:00<00:00, 19.13it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00021-000

Predicting BraTS-PED-00026-000:

In [ ]:
!ls /content/pred_fold0_noTTA


BraTS-PED-00008-000.nii.gz  BraTS-PED-00099-000.nii.gz
BraTS-PED-00021-000.nii.gz  BraTS-PED-00101-000.nii.gz
BraTS-PED-00026-000.nii.gz  BraTS-PED-00104-000.nii.gz
BraTS-PED-00042-000.nii.gz  BraTS-PED-00107-000.nii.gz
BraTS-PED-00050-000.nii.gz  BraTS-PED-00110-000.nii.gz
BraTS-PED-00055-000.nii.gz  BraTS-PED-00115-000.nii.gz
BraTS-PED-00063-000.nii.gz  BraTS-PED-00118-000.nii.gz
BraTS-PED-00078-000.nii.gz  BraTS-PED-00132-000.nii.gz
BraTS-PED-00079-000.nii.gz  dataset.json
BraTS-PED-00084-000.nii.gz  plans.json
BraTS-PED-00086-000.nii.gz  predict_from_raw_data_args.json
BraTS-PED-00096-000.nii.gz


In [ ]:
!mkdir -p /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT40epoch_80pruned_noTTA
!cp -r /content/pred_fold0_noTTA/* /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT40epoch_80pruned_noTTA/


فرآیند های سنجش
